# 14.2 高级提示技术 (Advanced Prompting)

高级提示技术通过更复杂的提示策略进一步提升模型性能，适用于复杂推理、多步骤任务和工具调用等场景。

## 技术总览

| 技术 | 核心思想 | 适用场景 |
|------|----------|----------|
| Self-Consistency | 多次采样取多数投票 | 提高推理可靠性 |
| ReAct | 推理+行动交替 | 需要工具调用的任务 |
| Directional Stimulus | 提示中嵌入引导信号 | 控制输出风格/方向 |
| Tree of Thought | 多路径探索+回溯 | 复杂规划问题 |
| Reflexion | 自我反思+改进 | 迭代优化输出 |
| DSPy | 编程式提示优化 | 系统化提示工程 |

## 1. 自我一致性（Self-Consistency）

自我一致性通过多次采样CoT推理路径，取多数投票作为最终答案，显著提升推理任务的准确率。

### 核心思想
- 同一问题可以有多条正确的推理路径
- 正确答案的推理路径更可能收敛到同一答案
- 错误答案的推理路径更分散
- 多数投票自然选择最一致的答案

### 实现要点
- 采样温度：0.5-1.0，太低缺乏多样性，太高噪声太多
- 采样次数：通常5-40次，越多越准但成本越高
- 投票策略：简单多数投票或加权投票（按log概率加权）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

torch.manual_seed(42)

class SelfConsistencyVoting:
    def __init__(self, model, n_answers=5):
        self.model = model
        self.n_answers = n_answers

    def sample_with_reasoning(self, query, n_samples=20, temperature=0.7):
        results = []
        for _ in range(n_samples):
            logits = self.model(query) / temperature
            noise = torch.randn_like(logits) * 0.3
            probs = F.softmax(logits + noise, dim=-1)
            answer = probs.argmax(dim=-1).item()
            log_prob = F.log_softmax(logits, dim=-1)[0, answer].item()
            results.append({'answer': answer, 'log_prob': log_prob, 'prob': probs[0, answer].item()})
        return results

    def majority_vote(self, results):
        answers = [r['answer'] for r in results]
        counts = Counter(answers)
        return counts.most_common(1)[0]

    def weighted_vote(self, results):
        answer_weights = {}
        for r in results:
            a = r['answer']
            answer_weights[a] = answer_weights.get(a, 0) + r['prob']
        best = max(answer_weights, key=answer_weights.get)
        return best, answer_weights[best] / sum(answer_weights.values())

class SimpleModel(nn.Module):
    def __init__(self, d=64, n_answers=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, n_answers))
    def forward(self, x):
        return self.net(x)

model = SimpleModel()
voter = SelfConsistencyVoting(model, n_answers=5)
query = torch.randn(1, 64)

results = voter.sample_with_reasoning(query, n_samples=30, temperature=0.8)

majority, count = voter.majority_vote(results)
weighted, confidence = voter.weighted_vote(results)

print('=== Self-Consistency Voting ===')
answers = [r['answer'] for r in results]
print(f'Sampled answers: {answers}')
print(f'\nMajority vote: answer={majority} (count={count}/{len(results)})')
print(f'Weighted vote: answer={weighted} (confidence={confidence:.2%})')

print(f'\nVote distribution:')
for answer, count in sorted(Counter(answers).items()):
    bar = '#' * count
    print(f'  Answer {answer}: {bar} ({count})')

print(f'\nKey: Weighted voting uses probability as confidence, more robust than simple majority.')

## 2. ReAct（Reasoning + Acting）

ReAct结合推理（Reasoning）和行动（Acting），模型交替进行思考和工具调用，适合需要外部信息的复杂任务。

### ReAct流程
```
Question → Thought → Action → Observation → Thought → Action → ... → Answer
```

### ReAct vs 纯推理 vs 纯行动
| 方法 | 优点 | 缺点 |
|------|------|------|
| 纯推理(CoT) | 无需外部工具 | 可能产生幻觉 |
| 纯行动 | 获取真实信息 | 缺乏推理能力 |
| ReAct | 推理+行动结合 | 步骤多、成本高 |

### ReAct的工业实践
- **工具设计**：每个工具的描述要清晰，包括输入输出格式
- **最大步数**：限制ReAct循环次数，防止无限循环
- **错误处理**：工具调用失败时的回退策略
- **观察截断**：过长的观察结果需要摘要

In [ ]:
class ReActAgent:
    def __init__(self, model, tools, d=64, max_steps=5):
        self.model = model
        self.tools = tools
        self.d = d
        self.max_steps = max_steps
        self.trajectory = []

    def think(self, context_emb):
        with torch.no_grad():
            logits = self.model(context_emb)
            thought_type = logits.argmax(dim=-1).item()
        return thought_type

    def act(self, tool_name, tool_input):
        if tool_name in self.tools:
            result = self.tools[tool_name](tool_input)
        else:
            result = f'Error: Tool {tool_name} not found'
        return result

    def run(self, question_emb, question_text=''):
        context = question_emb.clone()
        self.trajectory = [{'role': 'question', 'content': question_text}]

        for step in range(self.max_steps):
            thought_type = self.think(context)

            if thought_type < len(self.tools):
                tool_names = list(self.tools.keys())
                tool_name = tool_names[thought_type % len(tool_names)]
                observation = self.act(tool_name, context)
                obs_emb = torch.randn(1, self.d) * 0.1
                context = torch.cat([context, obs_emb], dim=1) if context.dim() > 1 else context
                self.trajectory.append({
                    'role': 'action',
                    'tool': tool_name,
                    'observation': 'simulated_result'
                })
            else:
                self.trajectory.append({
                    'role': 'answer',
                    'content': 'final_answer'
                })
                break

        if self.trajectory[-1]['role'] != 'answer':
            self.trajectory.append({'role': 'answer', 'content': 'max_steps_reached'})

        return self.trajectory

tools = {
    'search': lambda x: 'search_results',
    'calculate': lambda x: 'calculation_result',
    'lookup': lambda x: 'lookup_result',
}

class ReActModel(nn.Module):
    def __init__(self, d=64, n_actions=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, n_actions))
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        return self.net(x[:, :self.net[0].in_features] if x.shape[-1] > self.net[0].in_features else x)

react_model = ReActModel()
agent = ReActAgent(react_model, tools, d=64, max_steps=5)
question_emb = torch.randn(1, 64)

trajectory = agent.run(question_emb, 'What is the population of Paris?')

print('=== ReAct Agent ===')
for i, step in enumerate(trajectory):
    role = step['role'].upper()
    if role == 'QUESTION':
        print(f'  [{role}] {step["content"]}')
    elif role == 'ACTION':
        print(f'  [THOUGHT] I should use {step["tool"]}')
        print(f'  [ACTION] {step["tool"]}()')
        print(f'  [OBSERVATION] {step["observation"]}')
    elif role == 'ANSWER':
        print(f'  [{role}] {step["content"]}')

print(f'\nTotal steps: {len(trajectory)}')
print(f'\nKey: ReAct alternates between reasoning and tool use.')
print(f'Max steps limit prevents infinite loops.')

## 3. 方向性刺激提示（Directional Stimulus Prompting）

方向性刺激提示在提示中嵌入引导信号，控制模型的输出方向和风格，而不改变模型参数。

### 核心思想
- 在提示中加入"提示中的提示"，引导模型关注特定方面
- 例如：在摘要任务中加入"关注技术细节"，模型会更侧重技术内容
- 可以通过训练学习最优的方向性刺激

### 与Soft Prompting的区别
- 方向性刺激：在文本提示中添加引导关键词
- Soft Prompting：在嵌入空间添加可学习向量

In [ ]:
class DirectionalStimulus:
    def __init__(self, model, d=64, vocab_size=100):
        self.model = model
        self.d = d
        self.vocab_size = vocab_size
        self.stimulus_embeddings = nn.ParameterDict({
            'technical': nn.Parameter(torch.randn(d) * 0.02),
            'creative': nn.Parameter(torch.randn(d) * 0.02),
            'concise': nn.Parameter(torch.randn(d) * 0.02),
            'detailed': nn.Parameter(torch.randn(d) * 0.02),
        })

    def apply_stimulus(self, input_emb, stimulus_type='technical', alpha=0.3):
        stimulus = self.stimulus_embeddings[stimulus_type]
        stimulated = input_emb + alpha * stimulus.unsqueeze(0).unsqueeze(0).expand_as(input_emb)
        return stimulated

    def generate_with_stimulus(self, x, stimulus_type='technical', alpha=0.3, max_new=5):
        input_emb = self.model.embed(x)
        stimulated = self.apply_stimulus(input_emb, stimulus_type, alpha)
        h = self.model.transformer(stimulated)
        logits = self.model.head(h)

        generated = x.clone()
        for _ in range(max_new):
            next_logits = logits[:, -1, :]
            probs = F.softmax(next_logits, dim=-1)
            next_token = probs.multinomial(1)
            generated = torch.cat([generated, next_token], dim=1)
            input_emb = self.model.embed(generated)
            stimulated = self.apply_stimulus(input_emb, stimulus_type, alpha)
            h = self.model.transformer(stimulated)
            logits = self.model.head(h)

        return generated

class PromptModelWithEmbed(nn.Module):
    def __init__(self, d=64, vocab_size=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 2, d * 4, batch_first=True), 1
        )
        self.head = nn.Linear(d, vocab_size)
    def forward(self, x):
        h = self.embed(x)
        h = self.transformer(h)
        return self.head(h)

model = PromptModelWithEmbed()
ds = DirectionalStimulus(model, d=64, vocab_size=100)

x = torch.randint(0, 100, (1, 5))

print('=== Directional Stimulus Prompting ===')
for stim_type in ['technical', 'creative', 'concise', 'detailed']:
    out = ds.generate_with_stimulus(x, stimulus_type=stim_type, alpha=0.3, max_new=3)
    print(f'{stim_type:10s}: input={x.tolist()}, output={out[0, -3:].tolist()}')

print(f'\nKey: Different stimuli steer the model toward different output styles.')
print(f'Alpha controls stimulus strength: too high distorts, too low has no effect.')

## 4. Tree of Thought (ToT)

Tree of Thought将推理过程组织为树结构，允许探索多条推理路径并回溯错误路径，适合复杂的规划和推理任务。

### ToT vs CoT
- **CoT**：单条线性推理路径，无法回溯
- **ToT**：树状推理，可以探索多条路径，评估后选择最优

### ToT核心步骤
1. **生成**：为当前状态生成多个候选思考步骤
2. **评估**：评估每个候选步骤的价值
3. **搜索**：选择最有希望的路径继续探索（BFS或DFS）
4. **回溯**：如果当前路径不理想，回退到之前的状态

In [ ]:
class TreeOfThought:
    def __init__(self, model, d=64, n_candidates=3, max_depth=4):
        self.model = model
        self.d = d
        self.n_candidates = n_candidates
        self.max_depth = max_depth

    def generate_candidates(self, state_emb):
        candidates = []
        for _ in range(self.n_candidates):
            noise = 0.1 * torch.randn_like(state_emb)
            candidate = state_emb + noise
            candidates.append(candidate)
        return candidates

    def evaluate_state(self, state_emb):
        with torch.no_grad():
            if state_emb.dim() == 1:
                state_emb = state_emb.unsqueeze(0)
            score = torch.sigmoid(self.model.net(state_emb[:, :32])).mean().item()
        return score

    def search_bfs(self, initial_state, beam_width=2):
        beam = [(initial_state, 0.0, [])]
        best_path = None
        best_score = float('-inf')

        for depth in range(self.max_depth):
            all_candidates = []
            for state, score, path in beam:
                candidates = self.generate_candidates(state)
                for cand in candidates:
                    cand_score = self.evaluate_state(cand)
                    all_candidates.append((cand, cand_score, path + [depth]))

            all_candidates.sort(key=lambda x: x[1], reverse=True)
            beam = all_candidates[:beam_width]

            if beam[0][1] > best_score:
                best_score = beam[0][1]
                best_path = beam[0][2]

        return best_path, best_score

class ToTModel(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):
        return self.net(x)

tot_model = ToTModel()
tot = TreeOfThought(tot_model, d=64, n_candidates=3, max_depth=4)

initial = torch.randn(64)
best_path, best_score = tot.search_bfs(initial, beam_width=2)

print('=== Tree of Thought ===')
print(f'Initial state dim: {initial.shape}')
print(f'Best path: {best_path}')
print(f'Best score: {best_score:.4f}')
print(f'Search config: {tot.n_candidates} candidates/step, depth={tot.max_depth}, beam=2')

print(f'\nKey: ToT explores multiple reasoning paths and backtracks from dead ends.')
print(f'Beam search balances exploration with computational cost.')

## 5. 高级提示技术选择指南

### 根据任务选择
| 任务类型 | 推荐技术 | 原因 |
|----------|----------|------|
| 数学推理 | Self-Consistency | 多路径验证提高准确率 |
| 信息检索 | ReAct | 需要工具获取外部信息 |
| 创意生成 | Directional Stimulus | 控制创意方向 |
| 复杂规划 | Tree of Thought | 需要探索和回溯 |
| 代码生成 | CoT + Self-Consistency | 推理+验证 |

### 成本与效果权衡
| 技术 | 额外API调用 | 准确率提升 | 适用规模 |
|------|-------------|-----------|----------|
| Self-Consistency | 5-40x | +5-15% | 中小规模 |
| ReAct | 3-10x | +10-30% | 需要工具时 |
| ToT | 10-100x | +10-20% | 小规模复杂问题 |
| Directional Stimulus | 1x | +3-10% | 任何规模 |

### 组合使用
- **ReAct + Self-Consistency**：工具调用后多次验证
- **CoT + Directional Stimulus**：推理时引导关注重点
- **ToT + ReAct**：在树的每个节点使用工具

### 工业实践注意事项
1. **成本控制**：高级技术增加API调用次数，需要权衡成本和效果
2. **延迟管理**：多步推理增加响应时间，需要设置超时
3. **缓存策略**：相同子问题的结果可以缓存
4. **降级策略**：高级技术失败时回退到简单方法
5. **评估驱动**：用A/B测试验证高级技术是否真正提升业务指标